In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("product", StringType()),
    StructField("price", DoubleType()),
    StructField("quantity", IntegerType()),
    StructField("timestamp", StringType())
])

In [0]:
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker1:9092")
    .option("subscribe" ,"orders_topic")
    .option("startingOffsets", "latest")
    .load()
)

In [0]:
orders_df = (
    kafka_df
    .selectExpr("CAST(value AS STRING)")
    .select(from_json(col("value"), schema).alias("data"))
    .select("data.*")
    .withColumn("order_time", to_timestamp("timestamp"))
)

In [0]:
orders_df.writeStream \
  .format("delta") \
  .outputMode("append") \
  .trigger(availableNow=True) \
  .option("checkpointLocation","/Volumes/workspace/default/etl_volume/checkpoints/bronze_orders") \
  .toTable("workspace.default.bronze_orders")